# RQ2 starter — Early → later transition-network change

Starter notebook generated from the finalized RQ1 handoff. It does not claim a final RQ2 metric until that metric is pre-specified.

This starter loads the frozen RQ1 handoff: `rq1_merged_spm_quade_outputs/rq1_objects_for_rq2_rq3.pkl`.


# RQ2 starter — Early → later transition-network change

**Research question:** How does the transition network change from the early window to the later window, and how does that change differ between Later-Struggle and No-Later-Struggle histories?

### Planned comparison

1. Load the frozen RQ1 handoff.
2. Reuse the frozen seven-state labels and selected `k`.
3. Construct **early transitions** from the first `k` classified attempts.
4. Construct **later transitions** from the attempts after `k`.
5. Build outcome-specific transition-probability matrices for Later Struggle and No Later Struggle.
6. Quantify early→later network change using a pre-specified network-distance/stability measure.
7. Compare the magnitude and direction of network change between the two outcome groups.

**Important:** This starter does not choose a final network-change metric for you. The final RQ2 notebook should pre-register the metric and whether it is computed on raw transition probabilities, normalized matrices, or another representation before testing group differences.


In [ ]:
# ================================================================
# RQ2 STARTER CODE
# ================================================================
from pathlib import Path
import pickle
import numpy as np
import pandas as pd

HANDOFF_PATH = Path("rq1_merged_spm_quade_outputs/rq1_objects_for_rq2_rq3.pkl")

with open(HANDOFF_PATH, "rb") as f:
    handoff = pickle.load(f)

episode = handoff["episode"].copy()
population = handoff["opt_population"].copy()
selected_k = handoff["selected_k"]
STATE_ORDER = handoff["state_order"]
ALL_TRANSITIONS = handoff["all_transitions"]

print("Loaded RQ1 handoff.")
print("Selected k:", selected_k)
print("Histories:", len(population))

# ------------------------------------------------
# RQ2.1 Construct early and later transitions
# ------------------------------------------------
ep = episode.sort_values(
    ["userId", "skill", "start_time", "episode_row_order"],
    kind="mergesort"
).copy()

ep["skill_attempt"] = ep.groupby(["userId", "skill"]).cumcount() + 1
ep["outcome"] = ep.set_index(["userId", "skill"]).index.map(
    population.set_index(["userId", "skill"])["later_struggle"]
)

early = ep.loc[ep["skill_attempt"] <= selected_k].copy()
later = ep.loc[ep["skill_attempt"] > selected_k].copy()

def make_transition_table(frame):
    x = frame.sort_values(
        ["userId", "skill", "skill_attempt"],
        kind="mergesort"
    ).copy()
    x["next_state"] = x.groupby(["userId", "skill"])["state"].shift(-1)
    x = x.loc[x["next_state"].notna()].copy()
    x["transition"] = x["state"].astype(str) + " → " + x["next_state"].astype(str)
    return x

early_transitions = make_transition_table(early)
later_transitions = make_transition_table(later)

print("Early transitions:", len(early_transitions))
print("Later transitions:", len(later_transitions))

# ------------------------------------------------
# RQ2.2 Outcome-specific transition probabilities
# ------------------------------------------------
def transition_probability_matrix(transitions, outcome_value):
    sub = transitions.loc[transitions["outcome"] == outcome_value].copy()
    counts = pd.crosstab(sub["state"], sub["next_state"])
    counts = counts.reindex(index=STATE_ORDER, columns=STATE_ORDER, fill_value=0)
    row_totals = counts.sum(axis=1).replace(0, np.nan)
    return counts.div(row_totals, axis=0)

early_tna_NS = transition_probability_matrix(early_transitions, 0)
early_tna_S  = transition_probability_matrix(early_transitions, 1)
later_tna_NS = transition_probability_matrix(later_transitions, 0)
later_tna_S  = transition_probability_matrix(later_transitions, 1)

display(early_tna_NS.round(4))
display(early_tna_S.round(4))
display(later_tna_NS.round(4))
display(later_tna_S.round(4))

# ------------------------------------------------
# RQ2.3 STARTER: quantify early → later network change
# ------------------------------------------------
# TODO: Pre-specify the final network-change metric before inference.
# Examples to evaluate in the final RQ2 notebook:
#   - Frobenius distance between early and later probability matrices
#   - Jensen–Shannon distance after a clearly specified normalization
#   - edge-wise absolute change followed by a multiple-testing procedure
#
# Do not mix several metrics into the primary result after seeing which
# one produces the strongest group difference.

def frobenius_network_change(early_matrix, later_matrix):
    diff = later_matrix.fillna(0).to_numpy() - early_matrix.fillna(0).to_numpy()
    return float(np.sqrt(np.square(diff).sum()))

rq2_network_change = pd.DataFrame({
    "outcome_group": ["No Later Struggle", "Later Struggle"],
    "early_to_later_frobenius_change": [
        frobenius_network_change(early_tna_NS, later_tna_NS),
        frobenius_network_change(early_tna_S, later_tna_S),
    ],
})

display(rq2_network_change)
print("RQ2 starter complete: final inference/metric specification remains to be pre-specified.")
